# รัน `interleaved-reasoning` บน Google Colab

Notebook นี้**ต่อการทดลองจากที่ค้างใน repo** แล้วรันเมทริกซ์ที่เหลือจนจบ (resume ได้เสมอ)

> ⚠️ **repo นี้เป็น private** → ต้องมี GitHub token (PAT) **ตั้งแต่เซลล์แรกที่ clone** ไม่ใช่แค่ตอน push

**ลำดับการรัน (Runtime → Run all ได้เลย):**

1. เซลล์ **TOKEN** — ใส่ PAT 1 ใน 3 วิธี (พิมพ์ในเซลล์ / Colab Secrets 🔑 / ถามตอนรัน) → ใช้ทั้ง clone และ push
2. เซลล์ **SETUP** — clone ด้วย token, ติดตั้ง deps, โชว์ตารางสถานะ 33 runs
3. **smoke test** — เทรนจิ๋ว 30 steps ยืนยันความพร้อม
4. **เมทริกซ์** — resume `C_mlp_main` + รันที่เหลือ 30 รัน (~2.5–3.5 ชม., ข้ามอัน done เอง)
5. **pytest → render FINDINGS → บันทึกผล** (push กลับ branch เดิม)

ใช้ runtime **CPU ฟรี** (โค้ดล็อก CPU เพื่อ reproducibility) • ทำงานบนดิสก์ VM `/content` (เร็ว) แล้วสำรองขึ้น Drive ระหว่างทาง — ไม่เทรนบน Drive ตรง ๆ

In [ ]:
# ============================================================
# TOKEN — รันเซลล์นี้ "ก่อนเซลล์อื่นทั้งหมด"
# repo เป็น private → ต้องใช้ token ตั้งแต่ขั้น clone (ไม่ใช่แค่ตอน push)
# วิธีให้ token เลือก 1 ใน 3 วิธี:
#   1) วาง PAT ลงตัวแปร TOKEN ด้านล่าง (ค่าอยู่แค่ใน runtime นี้ — ห้าม save notebook ที่ฝัง token กลับขึ้น GitHub)
#   2) Colab Secrets: แถบซ้าย 🔑 → Add new secret → Name: GITHUB_TOKEN + เปิดสวิตช์ "Notebook access" (แนะนำ)
#   3) ปล่อยว่างไว้ → เซลล์นี้จะถามแบบซ่อนอักษรตอนรัน
# สร้าง PAT: github.com → Settings → Developer settings → Personal access tokens
#   fine-grained: เลือก repo นี้ + Contents: Read/Write   |   classic: scope repo
# ============================================================
TOKEN = ""   # ← วิธี 1: วาง PAT ตรงนี้ เช่น "github_pat_xxx" หรือ "ghp_xxx"

import getpass
from urllib.parse import quote

REPO_HOST_PATH = "github.com/gunnsmart/cot-hybrid-.git"   # ใช้ทั้ง clone และ push
SRC = None
if not TOKEN:
    try:
        from google.colab import userdata
        try:
            TOKEN = (userdata.get("GITHUB_TOKEN") or "").strip()
            if TOKEN:
                SRC = "Colab Secrets (GITHUB_TOKEN)"
        except Exception:  # SecretNotFoundError / NotebookAccessError / อื่น ๆ
            print("(ไม่มี secret GITHUB_TOKEN หรือยังไม่ได้เปิด Notebook access → ลองถามตอนรันแทน)")
    except ImportError:
        pass
if not TOKEN:
    try:
        TOKEN = getpass.getpass("GitHub PAT (จำเป็น — repo เป็น private): ").strip()
        if TOKEN:
            SRC = "ป้อนเองตอนรัน"
    except Exception:
        TOKEN = ""

if not TOKEN:
    raise RuntimeError(
        "ไม่มี GitHub token — repo นี้เป็น private จึง clone ไม่ได้เลย\n"
        "→ วาง PAT ในตัวแปร TOKEN ด้านบน หรือตั้ง Colab Secrets ชื่อ GITHUB_TOKEN แล้วรันเซลล์นี้ใหม่")

URL  = f"https://{quote(TOKEN, safe='')}@{REPO_HOST_PATH}"
PUSH = True
print(f"TOKEN OK (จาก {SRC or 'พิมพ์ในเซลล์'}) — จะใช้ token นี้ทั้งตอน clone และ push (ไม่มีพิมพ์ค่าออกจอ)")

In [ ]:
# ============================================================
# SETUP — clone (ด้วย token) + deps + ตารางสถานะ  |  รันหลังเซลล์ TOKEN
# ============================================================
import json, os, subprocess, sys

BRANCH   = "arena/01a0ede9-cot-hybrid"
BASE     = os.environ.get("WORK_BASE", "/content")   # ดิสก์ VM (เร็ว) — ของถาวรสำรองลง Drive แทน
REPO_DIR = os.path.join(BASE, "cot-hybrid-")
WORK_DIR = os.path.join(REPO_DIR, "interleaved-reasoning")
PY       = sys.executable
os.makedirs(BASE, exist_ok=True)

if "URL" not in globals():
    raise RuntimeError("รันเซลล์ TOKEN (เซลล์บนสุดของโค้ด) ก่อน — เซลล์นี้ใช้ token จากมันตอน clone")

# ---- (แนะนำ) ต่อ Drive ไว้สำรองผล — ยกเลิก/พลาดก็ทำงานต่อได้ (เซฟด้วย zip ท้าย notebook แทน)
DRIVE_DIR = None
try:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        print("กำลังขอเชื่อม Google Drive (ไว้สำรองผล) ... ถ้าไม่อยากต่อ กดยกเลิกในหน้าอนุญาตได้")
        drive.mount("/content/drive")
    if os.path.isdir("/content/drive/MyDrive"):
        DRIVE_DIR = "/content/drive/MyDrive/cot-hybrid-results"
        os.makedirs(DRIVE_DIR, exist_ok=True)
except Exception as e:
    print(f"(ทำงานต่อแบบไม่มี Drive — {type(e).__name__}: จะเซฟผลด้วย zip ในเซลล์ท้าย)")

# ---- clone ด้วย token (ครั้งแรก) หรืออัปเดตเป็น state ล่าสุด (ครั้งถัด ๆ ไป)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("clone ด้วย token →", REPO_DIR)
    r = subprocess.run(["git", "clone", "-b", BRANCH, URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            "clone ไม่สำเร็จ — ตรวจว่า PAT ถูกต้องและมีสิทธิ์ repo นี้\n"
            "(fine-grained: Contents Read/Write | classic: scope repo)\n"
            + (r.stderr or "").strip()[-400:])
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], capture_output=True, text=True)
    r = subprocess.run(["git", "-C", REPO_DIR, "merge", "--ff-only", f"origin/{BRANCH}"],
                       capture_output=True, text=True)
    print("อัปเดต repo:", (r.stdout or r.stderr).strip() or "up to date")

# ---- ยืนยันว่า token ใช้ได้จริง + branch มีอยู่
v = subprocess.run(["git", "-C", REPO_DIR, "ls-remote", "origin", f"refs/heads/{BRANCH}"],
                   capture_output=True, text=True)
if v.returncode != 0 or not v.stdout.strip():
    raise RuntimeError("เข้าถึง branch ไม่ได้ — token ไม่มีสิทธิ์ repo นี้ หรือ branch ถูกลบ\n" + (v.stderr or "")[-300:])
print(f"เข้าถึง repo OK (branch {BRANCH}) — push ภายหลังใช้ token เดียวกัน")

for cfg in (("user.name", "colab-runner"), ("user.email", "colab-runner@users.noreply.github.com")):
    subprocess.run(["git", "-C", REPO_DIR, "config", *cfg], check=False)

subprocess.run([PY, "-m", "pip", "install", "-q", "pytest"], check=False)

def run_status(rid):
    p = os.path.join(WORK_DIR, "results", rid + ".json")
    if not os.path.exists(p):
        return "-"
    try:
        return json.load(open(p)).get("status", "?")
    except Exception:
        return "?"

ALL_RUNS = (["A_mlp_main", "B_mlp_main", "C_mlp_main"]
            + [f"{t}_{f}_base" for t in "ABC" for f in ("emit", "latent")]
            + ["A_abl_noprice", "A_abl_nocommit", "A_abl_nondeg", "A_abl_bare",
               "A_abl_sig0", "A_abl_reset", "A_abl_redundant", "A_abl_nogate"]
            + [f"A_scale_n{n}" for n in (4, 8, 12, 24, 48)]
            + ["A_gru", "A_ssm", "A_transformer",
               "A_gru_emit", "A_gru_latent", "A_ssm_emit", "A_ssm_latent",
               "A_tf_emit", "A_tf_latent", "B_gru", "B_transformer"])

os.chdir(WORK_DIR)
done = sum(run_status(r) == "done" for r in ALL_RUNS)
print(f"\nสถานะ: {done}/{len(ALL_RUNS)} runs เสร็จแล้ว")
for r in ALL_RUNS:
    print(f"  {r:18s} {run_status(r)}")
print("\nWORK_DIR  =", WORK_DIR)
print("DRIVE_DIR =", DRIVE_DIR or "(ไม่มี — จะเซฟด้วย zip)")
print("\nSETUP OK — รันเซลล์ถัดไปได้")

In [ ]:
# ============================================================
# ทดสอบระบบ — เทรนจิ๋ว 30 steps (~<1 นาที) ให้มั่นใจว่าพร้อมก่อนรันจริง
# ============================================================
import os, shutil, subprocess
try:
    os.chdir(WORK_DIR)
except NameError:
    raise RuntimeError("รันเซลล์ SETUP ก่อน")

r = subprocess.run([PY, "-m", "scripts.train", "--run-id", "_smoke", "--task", "recall",
                    "--arch", "mlp", "--d", "64", "--n", "4", "--depth-max", "3",
                    "--p-trivial", "0.0", "--steps", "30", "--eval-every", "30",
                    "--batch", "64", "--train-n", "512", "--no-git"],
                   capture_output=True, text=True)
print(r.stdout[-1200:])
if r.returncode != 0:
    print(r.stderr[-1500:])
    raise RuntimeError("smoke test ล้มเหลว — ดู error ด้านบน (อย่ารันเมทริกซ์ต่อจนแก้ได้)")
for junk in ("results/_smoke.json", "checkpoints/_smoke.pt"):
    if os.path.exists(junk):
        os.remove(junk)
shutil.rmtree("runs/_smoke", ignore_errors=True)
print("SMOKE OK — พร้อมรันเมทริกซ์ (ไปเซลล์ถัดไป)")

## รันเมทริกซ์ที่เหลือทั้งหมด (~2.5–3.5 ชม.)

สูตรเดียวกับ `scripts/run_matrix.sh` (recipe จาก calibration `A_cal27`: σ=0.1, gate q=0.30, expectation content):

1. `C_mlp_main` — **resume จาก step 500** (โหลด snapshot + นับ step จาก JSON ให้เอง)
2. baselines 6 รัน (`--force emit`/`--force latent` ครบ 3 tasks) — ใช้ตัดเกณฑ์ 4
3. ablations 8 รัน — พิสูจน์ 4 failure modes
4. scaling N = 4/8/12/24/48
5. generalization GRU / SSM / transformer (+ per-arch baselines, cross-task)

**ทนต่อการหลุด:** เสร็จรันไหน push/สำรองรันนั้น (push ทุก 100 steps ระหว่างเทรน) หลุดกลางทาง → รัน TOKEN + SETUP + เซลล์นี้ซ้ำ (อัน done ถูกข้าม, อันค้าง resume)
**อยากดูแผนโดยไม่เทรนจริง:** รัน `os.environ["MATRIX_DRY"] = "1"` ในเซลล์ใหม่ก่อน แล้วรันเซลล์นี้

In [ ]:
# ============================================================
# เมทริกซ์ไดรเวอร์ — ข้ามอัน done, resume อันค้าง, สำรองขึ้น Drive รันต่อรัน
# ============================================================
import json, os, shutil, subprocess, time
try:
    os.chdir(WORK_DIR)
except NameError:
    raise RuntimeError("รันเซลล์ SETUP ก่อน")

REC = ["--sigma", "0.1", "--lam-price", "0.2", "--lam-commit", "0.1", "--lam-nd", "20",
       "--reader-layers", "2", "--batch", "96", "--q-conf", "0.30", "--content",
       "expectation", "--push-every", "100"]

def A(steps, *extra):
    return ["--task", "arithmetic", "--arch", "mlp", "--d", "128", "--n", "12",
            "--depth-max", "4", "--p-trivial", "0.0", "--steps", str(steps), *REC, *extra]
def L(steps, *extra):
    return ["--task", "logic", "--arch", "mlp", "--d", "128", "--n", "12",
            "--depth-max", "5", "--steps", str(steps), *REC, *extra]
def R(steps, *extra):
    return ["--task", "recall", "--arch", "mlp", "--d", "128", "--n", "12",
            "--depth-max", "5", "--p-trivial", "0.0", "--steps", str(steps), *REC, *extra]
def G(arch, task="arithmetic", *extra):
    cmd = ["--task", task, "--arch", arch, "--d", "96", "--n", "12",
           "--depth-max", "4" if task == "arithmetic" else "5"]
    if task != "logic":
        cmd += ["--p-trivial", "0.0"]
    return cmd + ["--steps", "2000", *REC, *extra]
def S(n):
    return ["--task", "arithmetic", "--arch", "mlp", "--d", "96", "--n", str(n),
            "--depth-max", "4", "--p-trivial", "0.0", "--steps", "2000", *REC]

MATRIX = {
    "C_mlp_main":    R(3000),
    "A_emit_base":   A(3000, "--force", "emit"),
    "A_latent_base": A(3000, "--force", "latent"),
    "B_emit_base":   L(3000, "--force", "emit"),
    "B_latent_base": L(3000, "--force", "latent"),
    "C_emit_base":   R(3000, "--force", "emit"),
    "C_latent_base": R(3000, "--force", "latent"),
    "A_abl_noprice":   A(2500, "--lam-price", "0.0"),
    "A_abl_nocommit":  A(2500, "--lam-commit", "0.0"),
    "A_abl_nondeg":    A(2500, "--lam-nd", "0.0"),
    "A_abl_bare":      A(2500, "--lam-price", "0.0", "--lam-commit", "0.0", "--lam-nd", "0.0"),
    "A_abl_sig0":      A(2500, "--sigma", "0.0"),
    "A_abl_reset":     A(2500, "--semantics", "reset"),
    "A_abl_redundant": A(2500, "--p-trivial", "0.4"),
    "A_abl_nogate":    A(2500, "--q-conf", "0.0"),
    "A_scale_n4":  S(4),
    "A_scale_n8":  S(8),
    "A_scale_n12": S(12),
    "A_scale_n24": S(24),
    "A_scale_n48": S(48),
    "A_gru":         G("gru"),
    "A_ssm":         G("ssm"),
    "A_transformer": G("transformer"),
    "A_gru_emit":    G("gru", "arithmetic", "--force", "emit"),
    "A_gru_latent":  G("gru", "arithmetic", "--force", "latent"),
    "A_ssm_emit":    G("ssm", "arithmetic", "--force", "emit"),
    "A_ssm_latent":  G("ssm", "arithmetic", "--force", "latent"),
    "A_tf_emit":     G("transformer", "arithmetic", "--force", "emit"),
    "A_tf_latent":   G("transformer", "arithmetic", "--force", "latent"),
    "B_gru":         G("gru", "logic"),
    "B_transformer": G("transformer", "logic"),
}

def status(rid):
    p = os.path.join("results", rid + ".json")
    if os.path.exists(p):
        try:
            return json.load(open(p)).get("status")
        except Exception:
            return None
    return None

DRY = os.environ.get("MATRIX_DRY") == "1"
if DRY:
    print("DRY RUN — แสดงแผนเท่านั้น ไม่เทรนจริง\n")

T0, failed = time.time(), []
for rid, args in MATRIX.items():
    st = status(rid)
    if st == "done":
        print(f"SKIP   {rid} (done)")
        continue
    latest = os.path.join("runs", rid, "latest.pt")
    best = os.path.join("checkpoints", rid + ".pt")
    would_resume = st == "running" and (os.path.exists(latest) or os.path.exists(best))
    resume = []
    if st == "running" and not DRY:
        if not os.path.exists(latest) and os.path.exists(best):
            os.makedirs(os.path.join("runs", rid), exist_ok=True)
            shutil.copy(best, latest)
            print(f"SEED   {rid}: ทำ snapshot จาก best checkpoint เพื่อ resume")
        if os.path.exists(latest):
            resume = ["--resume"]
    if DRY:
        print(f"DRY    {rid}: {'resume ต่อจาก snapshot' if would_resume else 'รันใหม่'}")
        continue

    print(f"\n=== {rid} === [{(time.time()-T0)/60:.0f}m elapsed]", flush=True)
    cmd = [PY, "-m", "scripts.train", "--run-id", rid, *args, *resume]
    if not globals().get("PUSH", False):
        cmd.append("--no-git")
    r = subprocess.run(cmd)
    if r.returncode != 0:
        print(f"FAILED {rid} (exit {r.returncode})")
        failed.append(rid)
        if r.returncode == 2:
            print("push ล้มเหลว → หยุดตามกติกา repo: ตรวจ PAT/สิทธิ์ Contents:RW แล้วรัน TOKEN + SETUP + เซลล์นี้ใหม่ (อัน done จะถูกข้าม)")
            break
        continue
    if DRIVE_DIR:
        dst = os.path.join(DRIVE_DIR, "results")
        os.makedirs(dst, exist_ok=True)
        shutil.copy2(os.path.join("results", rid + ".json"), dst)
        ck = os.path.join("checkpoints", rid + ".pt")
        if os.path.exists(ck):
            os.makedirs(os.path.join(DRIVE_DIR, "checkpoints"), exist_ok=True)
            shutil.copy2(ck, os.path.join(DRIVE_DIR, "checkpoints"))
        print(f"สำรอง {rid} -> Drive")

print(f"\nจบ: {len(MATRIX)-len(failed)}/{len(MATRIX)} ok | failed: {failed or 'ไม่มี'} | {(time.time()-T0)/60:.0f} นาที")
if not DRY:
    print("ขั้นถัดไป: รันเซลล์ pytest → render FINDINGS → บันทึกผล")

In [ ]:
# ============================================================
# ชุดทดสอบ: 6 เกณฑ์ + negative results + ความสม่ำเสมอของตัวเลข
# (suite เขียวได้ก็ต่อเมื่อเมทริกซ์ข้างบนครบ — ถ้ายังไม่ครบ ดูรายชื่อที่ fail ได้จาก output)
# ============================================================
import os, subprocess
try:
    os.chdir(WORK_DIR)
except NameError:
    raise RuntimeError("รันเซลล์ SETUP ก่อน")
r = subprocess.run([PY, "-m", "pytest", "-q"])
if r.returncode != 0:
    print("\npytest ยังไม่เขียว — ถ้าเมทริกซ์ยังไม่ครบให้รันเซลล์เมทริกซ์ต่อก่อน แล้วกลับมารันเซลล์นี้")

In [ ]:
# ============================================================
# render FINDINGS.md (+ บล็อกผลใน README.md) จาก results/*.json
# ============================================================
import os, subprocess
try:
    os.chdir(WORK_DIR)
except NameError:
    raise RuntimeError("รันเซลล์ SETUP ก่อน")
r = subprocess.run([PY, "-m", "scripts.render_findings"], capture_output=True, text=True)
print(r.stdout or "", r.stderr or "")
assert r.returncode == 0, "render_findings ล้มเหลว"
print(open("FINDINGS.md").read()[:2000])

In [ ]:
# ============================================================
# บันทึกผล — commit + push กลับ branch เดิม (token เดียวกับตอน clone)
# ============================================================
import glob, os, shutil, subprocess, zipfile
try:
    os.chdir(WORK_DIR)
except NameError:
    raise RuntimeError("รันเซลล์ SETUP ก่อน")
if "PUSH" not in globals():
    raise RuntimeError("รันเซลล์ TOKEN ก่อน (เซลล์นี้ใช้ token จากมันตอน push)")

subprocess.run(["git", "add", "-A"], check=True)
subprocess.run(["git", "commit", "-m", "colab: matrix complete + FINDINGS rendered", "--allow-empty"], check=False)
r = subprocess.run(["git", "push", "origin", BRANCH], capture_output=True, text=True)
print((r.stdout or r.stderr).strip()[-500:])
if r.returncode != 0:
    print("push ไม่สำเร็จ (non-fast-forward? → รัน SETUP ใหม่เพื่อ ff-only merge แล้ว push ซ้ำ)")
else:
    print("PUSH OK — ผลทั้งหมดอยู่บน GitHub แล้ว")
# เผื่อไว้เสมอ: สำรองลง Drive + zip (ได้ไฟล์กลับมาถือไว้ด้วย)
if DRIVE_DIR:
    for sub in ("results", "checkpoints"):
        dst = os.path.join(DRIVE_DIR, sub)
        os.makedirs(dst, exist_ok=True)
        n = 0
        for f in glob.glob(os.path.join(sub, "*")):
            if os.path.isfile(f):
                shutil.copy2(f, dst)
                n += 1
        print(f"สำรอง {n} ไฟล์ -> {dst}")
zip_path = os.path.join(BASE, "cot-hybrid-results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for item in ("results", "checkpoints", "FINDINGS.md", "README.md"):
        if os.path.isfile(item):
            z.write(item)
        elif os.path.isdir(item):
            for root, _, fns in os.walk(item):
                for fn in fns:
                    z.write(os.path.join(root, fn))
print("zip สำรองพร้อม:", zip_path)

## ปัญหาที่พบบ่อย

| อาการ | สาเหตุ/วิธีแก้ |
|---|---|
| เซลล์ TOKEN ขึ้น `ไม่มี GitHub token` | repo เป็น private ต้องมี PAT: วางในตัวแปร `TOKEN` หรือตั้ง Colab Secrets ชื่อ `GITHUB_TOKEN` (+ เปิด Notebook access) แล้วรันเซลล์ใหม่ |
| `clone ไม่สำเร็จ` | PAT ผิด/หมดอายุ หรือไม่มีสิทธิ์ repo (fine-grained ต้อง Contents: Read/**Write**, classic ต้อง scope `repo`) |
| `RuntimeError: รันเซลล์ ... ก่อน` | runtime ถูกรีเซ็ต/รันข้ามลำดับ → ไล่รัน TOKEN → SETUP → เซลล์ถัดไปอีกครั้ง (ของที่ done ไม่หาย เพราะอยู่ในไฟล์ JSON ที่ push ไว้) |
| เซสชันหลุดกลางเมทริกซ์ (free tier idle 90 นาที / 12 ชม.) | เปิดใหม่ → TOKEN → SETUP → เซลล์เมทริกซ์ ต่อทันที อัน done ถูกข้าม อันค้าง resume |
| push fail (exit 2) ตอนเทรน | PAT ไม่มีสิทธิ์เขียน → สร้าง PAT ใหม่แบบ Contents: RW แล้วรัน TOKEN + SETUP + เมทริกซ์ซ้ำ |
| push บอก non-fast-forward | มี push จากที่อื่นทับหน้า → รัน SETUP (merge --ff-only ให้) แล้วกดเซลล์บันทึกผลซ้ำ |
| อยากรู้ก่อนว่าจะรันอะไรบ้าง | `os.environ["MATRIX_DRY"]="1"` แล้วรันเซลล์เมทริกซ์ = แสดงแผนอย่างเดียว |
| ทำไมไม่ใช้ GPU | `scripts/train.py` ล็อก `DEVICE="cpu"` เพื่อ reproducibility (noise ใช้ seeded generator) — โมเดลเล็ก เร็วพอบน CPU |

**ความปลอดภัยของ token:** แนะนำ Colab Secrets (เข้ารหัสฝั่ง Google, ไม่ลงโค้ด) / ถ้าพิมพ์ในเซลล์ อย่า save notebook ตัวนี้ทับขึ้น GitHub / token ไม่เคยถูก print ออกจอ และควร revoke เมื่อใช้เสร็จ